# PSF Estimation & Deconvolution Pipeline

This notebook drives the `psfselect` package, which wraps **EPFL's PSF Generator**
(a Java application) through a small Python wrapper. The flow is:

1. **Make the Java wrapper work** and smoke-test it (Part 1).
2. **Pick one `.tif` stack** and derive optical parameters from its metadata (Part 2).
3. **Compute a PSF with every estimation model** and compare them as 2D slices **and in 3D** (Part 3).
4. **Pick the nuclei channel** and run Richardson-Lucy deconvolution (Parts 4-6).
5. **Deconvolve the full nuclei volume** and view raw vs. deconvolved in **napari** (Parts 7-8).


## Setup (Google Colab)\n\nRun this once. It mounts your Google Drive, then **searches your Drive for\n`psfgenerator.jar`** to locate the `psf` folder automatically — so you don't\nhave to hard-code any path. It also installs the package and a **Java runtime**\n(the EPFL PSF Generator is a Java app; without a JRE the wrapper silently falls\nback to the pure-Python `psfmodels` backend).\n\nIf the search can't find the folder, the error lists your top-level Drive\nfolders so you can set `psf_pkg_dir` by hand.\n

In [3]:
import sys
import os
import glob

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

    # Since the FYP folder is in your Google Drive with the same structure:
    psf_pkg_dir = '/content/drive/MyDrive/FYP'

    if not os.path.exists(psf_pkg_dir):
        # Fallback if it's in a subfolder
        hits = glob.glob('/content/drive/MyDrive/**/FYP', recursive=True)
        if hits:
            psf_pkg_dir = hits[0]
        else:
            raise FileNotFoundError("Could not find the 'FYP' folder in your Google Drive. Please ensure you have a folder named 'FYP' in your Drive containing 'code/psf'.")

    os.chdir(psf_pkg_dir)

    # The EPFL PSF Generator is a JAVA application -> install a JRE so the
    # Python wrapper can actually run the JAR (otherwise it falls back to psfmodels).
    !apt-get -qq update && apt-get -qq install -y default-jre
    !pip install -q -r requirements.txt scikit-image
    !pip install -q -e .
else:
    # Run the notebook from the code/psf/ directory locally
    psf_pkg_dir = os.getcwd()
    if psf_pkg_dir not in sys.path:
        sys.path.append(psf_pkg_dir)

print("Working directory:", psf_pkg_dir)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 581.1/581.1 kB 11.1 MB/s eta 0:00:00a 0:00:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
ERROR: file:///content/drive/MyDrive/FYP does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.
Working directory: /content/drive/MyDrive/FYP


In [8]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

# The JAR file is located inside FYP/code/psf/ (which is the current directory psf_pkg_dir)
import os
os.environ["PSF_GENERATOR_JAR"] = os.path.join(psf_pkg_dir, "psfgenerator.jar")

from psfselect import MODELS, MODEL_LABELS
from psfselect.parameters import PSFParams, params_from_metadata
from psfselect.backends import render_psf, available_backends
from psfselect.metadata import validate_samples
from psfselect.metadata_leica import extract_metadata
from psfselect.visualize_napari import load_channels
from skimage.restoration import richardson_lucy

print("Known PSF models:", MODELS)


ModuleNotFoundError: No module named 'psfselect'

## 1. Make the Java wrapper work (and test it)

`available_backends()` probes whether the EPFL JAR can be located **and** whether a
Java runtime is on the `PATH`. We then force the `epfl` backend on a single small
PSF to prove the JAR actually computes and returns a volume.


In [ ]:
import shutil

print("java on PATH:", shutil.which("java"))
backends = available_backends()
print("Available backends:", backends)

assert backends["epfl"], (
    "EPFL JAR not available.\n"
    " - check PSF_GENERATOR_JAR points to psfgenerator.jar, and\n"
    " - check a Java runtime is installed (in Colab: apt-get install default-jre)."
)

# Smoke-test: force the Java backend on one model and confirm it really ran.
test_params = PSFParams(na=1.0, wavelength_nm=510.0, ni=1.33, ns=1.33,
                        voxel_xy_um=0.1, voxel_z_um=0.3, nx=31, nz=15)
vol, backend_used, cfg = render_psf("born_wolf", test_params, backend="epfl")

print(f"Rendered Born & Wolf via '{backend_used}' backend, shape={vol.shape}")
assert backend_used == "epfl", "Expected the EPFL Java backend to run the JAR!"
print("Java wrapper works.")

## 2. Pick a `.tif` and derive parameters from its metadata

We use `16012025_cmlc2_lifeactxnuclear_48hpf.lif - Series005.tif` (a 3-channel,
193-slice, 512x512 zebrafish cardiac stack). Rather than typing optical parameters
by hand, we extract them from the file and let documented defaults fill any gaps.


In [ ]:
# The dataset lives next to the psf folder at  code/data/raw/...  so we build the
# path relative to psf_pkg_dir -> works the same in Colab and locally.
rel = "../data/raw/cmlc2_lifeactXnuclear/48hpf/16012025_cmlc2_lifeactxnuclear_48hpf.lif - Series005.tif"
image_path = os.path.normpath(os.path.join(psf_pkg_dir, rel))
assert os.path.exists(image_path), f"Image not found: {image_path}"
print("Chosen file:", Path(image_path).name)

# Extract metadata -> resolve defaults -> build PSF params.
# nx/nz are kept small so every PSF (and the deconvolution) stays fast here.
meta, info = extract_metadata(image_path)
sample = validate_samples([meta])[0]
params = params_from_metadata(sample, nx=63, nz=31)

print("\nDerived PSF parameters:")
print(f"  NA       = {params.na}")
print(f"  ni / ns  = {params.ni} / {params.ns}")
print(f"  lambda   = {params.wavelength_nm} nm")
print(f"  voxel xy = {params.voxel_xy_um:.4f} um,  z = {params.voxel_z_um:.4f} um")
print(f"  grid     = nx {params.nx}, nz {params.nz}")
if sample.missing_fields:
    print(f"  [defaults applied for: {', '.join(sample.applied_defaults)}]")

## 3. Compute a PSF with every estimation model

`backend="auto"` runs the **EPFL Java JAR** for Born & Wolf, Gibson & Lanni and
Richards & Wolf, and uses the instant `psfmodels` approximation for Variable-RI
Gibson & Lanni (the JAR's VRIGL routine does not reliably terminate). The
`backend=` column below shows which engine produced each PSF.


In [ ]:
psfs = {}
for model in MODELS:
    vol, backend_used, cfg = render_psf(model, params, backend="auto")
    psfs[model] = vol
    print(f"{MODEL_LABELS[model]:52s} -> backend={backend_used:9s} shape={vol.shape}")

# Central z-slice of each PSF
fig, axes = plt.subplots(1, len(psfs), figsize=(4 * len(psfs), 4))
axes = np.atleast_1d(axes)
for ax, (model, vol) in zip(axes, psfs.items()):
    ax.imshow(vol[vol.shape[0] // 2], cmap="magma")
    ax.set_title(model, fontsize=10)
    ax.axis("off")
plt.tight_layout()
plt.show()

## 3b. Compare the PSFs in 3D

A single 2D slice hides the **axial elongation** that distinguishes these models. Here we
render an isosurface of each PSF (at 10% of its peak), scaled to physical microns so the
z-stretch is shown to scale.

In [ ]:
from skimage import measure
from mpl_toolkits.mplot3d.art3d import Poly3DCollection

ISO = 0.1                      # isosurface level, as a fraction of each PSF's peak
dz, dxy = params.voxel_z_um, params.voxel_xy_um

fig = plt.figure(figsize=(5 * len(psfs), 5))
for i, (model, vol) in enumerate(psfs.items()):
    ax = fig.add_subplot(1, len(psfs), i + 1, projection="3d")
    v = vol / max(float(vol.max()), 1e-12)
    try:
        verts, faces, _, _ = measure.marching_cubes(v, level=ISO)
        mesh = Poly3DCollection(verts[faces], alpha=0.4, facecolor="orchid", linewidths=0)
        ax.add_collection3d(mesh)
        ax.set_xlim(0, v.shape[0]); ax.set_ylim(0, v.shape[1]); ax.set_zlim(0, v.shape[2])
        ax.set_box_aspect((v.shape[0] * dz, v.shape[1] * dxy, v.shape[2] * dxy))
    except (RuntimeError, ValueError):
        m = v > ISO
        zz, yy, xx = np.where(m)
        ax.scatter(zz, yy, xx, c=v[m], cmap="magma", s=2, alpha=0.3)
    ax.set_title(model, fontsize=10)
    ax.set_xlabel("z"); ax.set_ylabel("y"); ax.set_zlabel("x")
plt.tight_layout()
plt.show()

## 4. Load the image and pick the nuclei channel

`load_channels` normalises any layout to `(C, Z, Y, X)`. This stack has 3 channels, so we
show the central slice of each one — identify the **nuclei** channel and set `NUCLEI_CHANNEL` below.

In [ ]:
raw, ch_idx = load_channels(image_path)   # (C, Z, Y, X)
print("Loaded volume (C, Z, Y, X):", raw.shape, " channels:", ch_idx)

# Central z-slice of every channel, to identify which one holds the nuclei
C, zc = raw.shape[0], raw.shape[1] // 2
fig, axes = plt.subplots(1, C, figsize=(5 * C, 5))
for ax, c in zip(np.atleast_1d(axes), range(C)):
    ax.imshow(raw[c, zc], cmap="gray")
    ax.set_title(f"channel {c}")
    ax.axis("off")
plt.tight_layout()
plt.show()

Set `NUCLEI_CHANNEL` to whichever channel above shows the nuclei. We crop a small
sub-volume of it for the quick 4-model comparison in Part 5; the **full, uncropped** nuclei
volume is used later for napari.

In [ ]:
NUCLEI_CHANNEL = 1            # <-- set to the nuclei channel index shown above

nuclei_full = raw[NUCLEI_CHANNEL]          # (Z, Y, X), full resolution


def center_crop(v, z=31, xy=192):
    nz, ny, nx = v.shape
    z, sy, sx = min(z, nz), min(xy, ny), min(xy, nx)
    z0, y0, x0 = (nz - z) // 2, (ny - sy) // 2, (nx - sx) // 2
    return v[z0:z0 + z, y0:y0 + sy, x0:x0 + sx]


image_crop = center_crop(nuclei_full, z=31, xy=192)
print("Cropped nuclei sub-volume (Z, Y, X):", image_crop.shape)

## 5. Richardson-Lucy deconvolution (10 iterations)

We run **10 iterations** of `skimage.restoration.richardson_lucy` with each
model's PSF. The image is normalised to `[0, 1]` and each PSF is given unit
energy (sum = 1) before deconvolution.


In [ ]:
NUM_ITERS = 10

img = image_crop.astype(float)
img /= max(img.max(), 1e-12)

deconvolved = {}
for model, psf in psfs.items():
    print(f"Richardson-Lucy ({NUM_ITERS} iters) with {model} PSF ...")
    # skimage does not normalise the PSF; give it unit energy (sum=1)
    psf_norm = psf / max(float(psf.sum()), 1e-12)
    deconvolved[model] = richardson_lucy(img, psf_norm, num_iter=NUM_ITERS)

print("Deconvolution finished!")

## 6. Results comparison

Central z-slice of the raw crop next to the deconvolution obtained with each PSF model.


In [ ]:
n = len(deconvolved) + 1
fig, axes = plt.subplots(1, n, figsize=(5 * n, 5))
zc = img.shape[0] // 2

axes[0].imshow(img[zc], cmap="gray")
axes[0].set_title("Raw (cropped)")
axes[0].axis("off")

for ax, (model, dec) in zip(axes[1:], deconvolved.items()):
    d = dec / max(float(dec.max()), 1e-12)
    ax.imshow(d[zc], cmap="gray")
    ax.set_title(f"RL: {model}", fontsize=10)
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Full-volume nuclei deconvolution (no crop)

Now the real thing: deconvolve the **entire** nuclei channel (e.g. 193x512x512), not a
patch. 10 Richardson-Lucy iterations take ~35 s on this machine.

In [ ]:
import time

DECONV_MODEL = "gibson_lanni"     # PSF model used for the full deconvolution

nuclei = nuclei_full.astype(float)
nuclei /= max(float(nuclei.max()), 1e-12)
psf_full = psfs[DECONV_MODEL] / psfs[DECONV_MODEL].sum()

print(f"Deconvolving FULL nuclei volume {nuclei.shape} with the {DECONV_MODEL} PSF "
      f"({NUM_ITERS} iters)...")
t0 = time.time()
nuclei_deconv = richardson_lucy(nuclei, psf_full, num_iter=NUM_ITERS)
print(f"Done in {time.time() - t0:.0f}s.  Output shape: {nuclei_deconv.shape}")

## 8. View before / after in napari (full 3D)

Opens napari with **only the nuclei channel**: raw vs. deconvolved, as full 3D volumes
(`ndisplay=3`), scaled to physical microns so the axial sampling is correct. napari is a
desktop app — run this locally, not in Colab. Close the window to return to the notebook.

In [ ]:
try:
    import napari
except ImportError as e:
    raise ImportError(
        "napari is not installed. Install it with:  pip install 'napari[all]'\n"
        "(napari is a desktop GUI app -- run this notebook locally, not in Colab)."
    ) from e

voxel = (params.voxel_z_um, params.voxel_xy_um, params.voxel_xy_um)   # (z, y, x) microns

viewer = napari.Viewer(ndisplay=3)
viewer.add_image(nuclei,        name="nuclei (raw)",          scale=voxel,
                 colormap="magenta", blending="additive")
viewer.add_image(nuclei_deconv, name="nuclei (deconvolved)",  scale=voxel,
                 colormap="green",   blending="additive")
napari.run()